# Analyze per Die Matrix

This notebook demonstrates how to find the **reference behavior** for each die matrix, compare individual pieces against it, and identify which process segments show the most variability.

All analysis is performed on the gold parquet dataset (clean, validated pieces with partial times).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

GOLD_PATH = Path('../data/gold/pieces.parquet')
df = pd.read_parquet(GOLD_PATH)

partial_cols = [
    'partial_furnace_to_2nd_strike_s',
    'partial_2nd_to_3rd_strike_s',
    'partial_3rd_to_4th_strike_s',
    'partial_4th_strike_to_auxiliary_press_s',
    'partial_auxiliary_press_to_bath_s'
]

lifetime_cols = [
    'lifetime_2nd_strike_s',
    'lifetime_3rd_strike_s',
    'lifetime_4th_strike_s',
    'lifetime_auxiliary_press_s',
    'lifetime_bath_s'
]

print(f"Loaded {len(df):,} pieces")
print(f"Die matrices: {sorted(df['die_matrix'].unique())}")

Loaded 168,059 pieces
Die matrices: [np.int64(4974), np.int64(5052), np.int64(5090), np.int64(5091)]


## 1. Reference profile per die matrix

The **reference (optimal) behavior** for each die matrix is defined by its median cumulative times. The median is more robust than the mean — it is not pulled by residual edge cases.

This table shows how long a typical piece takes to reach each stage, per matrix.

In [2]:
# 1. Reference profile - median cumulative times per matrix
reference = df.groupby('die_matrix')[lifetime_cols].median().round(2)
display(reference)

,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s
die_matrix,,,,,
4974,17.3,23.9,37.1,54.2,55.9
5052,18.3,25.3,39.3,56.7,58.3
5090,17.7,24.6,38.5,56.4,58.0
5091,18.5,25.6,38.2,57.5,59.1


## 2. Reference partial times per die matrix

The partial times (time spent **between** consecutive stages) are more diagnostically useful than cumulative times — they isolate each segment of the process.

In [3]:
# 2. Reference partial times per die matrix
reference_partials = df.groupby('die_matrix')[partial_cols].median().round(2)
display(reference_partials)

,partial_furnace_to_2nd_strike_s,partial_2nd_to_3rd_strike_s,partial_3rd_to_4th_strike_s,partial_4th_strike_to_auxiliary_press_s,partial_auxiliary_press_to_bath_s
die_matrix,,,,,
4974,17.3,6.5,13.1,17.0,1.8
5052,18.3,6.9,13.7,17.3,1.6
5090,17.7,6.8,13.8,17.7,1.6
5091,18.5,7.0,13.5,17.0,1.6


## 3. Variability per segment per die matrix

Which segments are most variable? High standard deviation relative to the median (coefficient of variation) indicates segments where the process is less stable — these are the primary candidates for delay investigation.

**CV = std / median × 100%** — higher CV means more variability relative to the typical value.

In [4]:
# 3. Variability per segment per die matrix (CV = std / median * 100)
cv_data = []
for matrix in sorted(df['die_matrix'].unique()):
    group = df[df['die_matrix'] == matrix]
    for col in partial_cols:
        median = group[col].median()
        std = group[col].std()
        cv = (std / median * 100) if median > 0 else None
        cv_data.append({'die_matrix': matrix, 'segment': col.replace('partial_', '').replace('_s', ''), 'cv_pct': round(cv, 2)})

cv_df = pd.DataFrame(cv_data)
display(cv_df.pivot(index='segment', columns='die_matrix', values='cv_pct'))

die_matrix,4974,5052,5090,5091
segment,,,,
2nd_to_3rdtrike,4.36,6.91,9.45,9.28
3rd_to_4thtrike,2.21,5.62,7.27,8.26
4thtrike_to_auxiliary_press,6.93,6.25,8.95,5.93
auxiliary_press_to_bath,2.70,3.07,5.41,5.29
furnace_to_2ndtrike,9.08,9.88,13.06,12.74


## 4. Deviation from reference per piece

For each piece, compute the deviation from its die matrix reference at each stage. Positive deviation = slower than reference. This allows identifying both slow individual pieces and systematic drifts.

In [5]:
# 4. Deviation from reference per piece
ref = df.groupby('die_matrix')[partial_cols].median()

for col in partial_cols:
    dev_col = col.replace('partial_', 'dev_').replace('_s', '_dev_s')
    df[dev_col] = df.apply(lambda row: row[col] - ref.loc[row['die_matrix'], col], axis=1)

dev_cols = [c for c in df.columns if c.startswith('dev_')]
print("Deviation columns added:")
display(df[['die_matrix'] + dev_cols].head())

Deviation columns added:


,die_matrix,dev_furnace_to_2nd_dev_strike_dev_s,dev_2nd_to_3rd_dev_strike_dev_s,dev_3rd_to_4th_dev_strike_dev_s,dev_4th_dev_strike_to_auxiliary_press_dev_s,dev_auxiliary_press_to_bath_dev_s
0,5052,-0.400000,-0.199999,-0.299999,-0.700001,0.000000
1,5052,-0.400000,-0.199999,-0.399998,-0.400002,0.000000
2,5052,-0.099998,-0.300001,-0.199999,-0.299999,0.000000
3,5052,0.100000,-0.199999,-0.399998,-0.200001,-0.000004
4,5052,-0.099998,-0.300001,-0.299997,0.000000,0.099998


## 5. Identify slow pieces and their penalized segment

A piece is considered **slow** if its total bath time exceeds the 90th percentile for its die matrix. For each slow piece, identify which segment contributed the most delay.

In [6]:
# 5. Identify slow pieces and penalized segment
p90 = df.groupby('die_matrix')['lifetime_bath_s'].quantile(0.9)
df['is_slow'] = df.apply(lambda row: row['lifetime_bath_s'] > p90[row['die_matrix']], axis=1)

# For each slow piece, find which segment contributed most delay
df['penalized_segment'] = df[dev_cols].idxmax(axis=1)
df['penalized_segment'] = df['penalized_segment'].where(df['is_slow'], None)

print(f"Total slow pieces: {df['is_slow'].sum():,} ({df['is_slow'].mean()*100:.1f}%)")
display(df[df['is_slow']][['die_matrix', 'lifetime_bath_s', 'penalized_segment']].head(10))

Total slow pieces: 16,390 (9.8%)


/var/folders/2t/259vktq9491gx6_y4jwx39pc0000gn/T/ipykernel_87304/3431874430.py:6: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  df['penalized_segment'] = df[dev_cols].idxmax(axis=1)


,die_matrix,lifetime_bath_s,penalized_segment
115,5052,67.099998,dev_4th_dev_strike_to_auxiliary_press_dev_s
226,5052,63.700001,dev_4th_dev_strike_to_auxiliary_press_dev_s
236,5052,64.699997,dev_4th_dev_strike_to_auxiliary_press_dev_s
500,5052,61.700001,dev_furnace_to_2nd_dev_strike_dev_s
944,5052,67.099998,dev_4th_dev_strike_to_auxiliary_press_dev_s
962,5052,62.000000,dev_2nd_to_3rd_dev_strike_dev_s
1074,5052,62.599998,dev_furnace_to_2nd_dev_strike_dev_s
1077,5052,62.000000,dev_furnace_to_2nd_dev_strike_dev_s
1079,5052,62.299999,dev_furnace_to_2nd_dev_strike_dev_s
1080,5052,61.799999,dev_furnace_to_2nd_dev_strike_dev_s


## 6. Slow pieces per die matrix

How slow pieces distribute across die matrices, and which segments are most often penalized per matrix.

In [7]:
# 6. Slow pieces per die matrix
slow_summary = df[df['is_slow']].groupby('die_matrix').agg(
    slow_pieces=('is_slow', 'sum'),
    most_penalized_segment=('penalized_segment', lambda x: x.value_counts().index[0])
).reset_index()

display(slow_summary)

,die_matrix,slow_pieces,most_penalized_segment
0,4974,1559,dev_furnace_to_2nd_dev_strike_dev_s
1,5052,2045,dev_furnace_to_2nd_dev_strike_dev_s
2,5090,7944,dev_furnace_to_2nd_dev_strike_dev_s
3,5091,4842,dev_furnace_to_2nd_dev_strike_dev_s


## 7. Time evolution — detecting drift

Does the process get slower over time for a given matrix? Plot the daily median bath time per matrix to detect progressive deterioration.

In [ ]:
# 7. Daily median bath time per matrix - detect drift
df['date'] = pd.to_datetime(df['timestamp']).dt.date

daily = df.groupby(['date', 'die_matrix'])['lifetime_bath_s'].median().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#457b9d', '#2a9d8f', '#e9c46a', '#f4a261']

for i, matrix in enumerate(sorted(df['die_matrix'].unique())):
    group = daily[daily['die_matrix'] == matrix]
    ax.plot(group['date'], group['lifetime_bath_s'], 
            label=f"Matrix {matrix}", color=colors[i], linewidth=1.5)

ax.set_xlabel('Date')
ax.set_ylabel('Median bath time (s)')
ax.set_title('Daily median bath time per die matrix — drift detection')
ax.legend()
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Segment variability ranking

Across all matrices, which process segment is the most unstable? This is where maintenance and process engineering should focus attention.

In [ ]:
# 8. Segment variability ranking across all matrices
overall_cv = []
for col in partial_cols:
    median = df[col].median()
    std = df[col].std()
    cv = std / median * 100
    overall_cv.append({'segment': col.replace('partial_', '').replace('_s', ''), 'cv_pct': round(cv, 2)})

cv_ranking = pd.DataFrame(overall_cv).sort_values('cv_pct', ascending=False)
display(cv_ranking)

## 9. Summary

Key findings from the per-matrix analysis.

In [ ]:
# 9. Summary
print("=" * 60)
print("PER-MATRIX ANALYSIS SUMMARY")
print("=" * 60)

print("\n DATASET OVERVIEW")
print(f"  Total pieces analyzed: {len(df):,}")
print(f"  Die matrices: {sorted(df['die_matrix'].unique())}")
print(f"  Date range: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

print("\n REFERENCE BATH TIMES (median per matrix)")
for matrix in sorted(df['die_matrix'].unique()):
    group = df[df['die_matrix'] == matrix]
    median = group['lifetime_bath_s'].median()
    print(f"  Matrix {matrix}: {median:.1f}s")

print("\n SLOW PIECES (above P90 bath time)")
for matrix in sorted(df['die_matrix'].unique()):
    group = df[df['die_matrix'] == matrix]
    slow = group['is_slow'].sum()
    total = len(group)
    print(f"  Matrix {matrix}: {slow:,} slow pieces ({slow/total*100:.1f}%)")

print("\n MOST PENALIZED SEGMENT PER MATRIX")
for matrix in sorted(df['die_matrix'].unique()):
    group = df[(df['die_matrix'] == matrix) & (df['is_slow'])]
    if len(group) > 0:
        top_segment = group['penalized_segment'].value_counts().index[0]
        top_count = group['penalized_segment'].value_counts().iloc[0]
        print(f"  Matrix {matrix}: {top_segment} ({top_count:,} pieces)")

print("\n SEGMENT VARIABILITY RANKING (CV across all matrices)")
for _, row in cv_ranking.iterrows():
    print(f"  {row['segment']}: {row['cv_pct']}% CV")

print("\n DRIFT DETECTION")
for matrix in sorted(df['die_matrix'].unique()):
    group = df[df['die_matrix'] == matrix].copy()
    group['date'] = pd.to_datetime(group['timestamp']).dt.date
    daily = group.groupby('date')['lifetime_bath_s'].median()
    if len(daily) > 1:
        drift = daily.iloc[-1] - daily.iloc[0]
        direction = "↑ increasing" if drift > 0.5 else "↓ decreasing" if drift < -0.5 else "→ stable"
        print(f"  Matrix {matrix}: {direction} ({drift:+.1f}s over active period)")

print("=" * 60)